# Anális de datos RUES

En este notebook realizamos una limpieza y análisis de los datos obtenidos mediante [web-scraping](https://github.com/pabloibargon/scraper).

In [1]:
# Imports
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

## Carga del dataset original

In [2]:
df = pd.read_csv('rues_20251109_042421.csv')

## Análisis de la calidad de los datos

In [3]:
# Resumen de columnas numéricas
display(df.describe().T)
# Resumen de columnas no numéricas
df.describe(exclude="number").T

,count,mean,std,min,25%,50%,75%,max
Número de Matrícula,4377.0,6.897020e+08,2.359404e+09,2.0,141816.0,17618504.0,38794512.0,9.000701e+09
Último año renovado,4377.0,1.708965e+03,7.196395e+02,0.0,1997.0,2010.0,2021.0,2.025000e+03


,count,unique,top,freq
Identificación,4377,3695,SIN IDENTIFICACION,335
Estado de la matrícula,4377,3,Activa,4371
nombre,4377,4377,COMERCIALIZADORA LA LLAMA S A,1
Categoria de la Matrícula,4377,3,Sociedad ó persona juridica principal ó esal,4362
Tipo de Sociedad,4377,5,Sociedad comercial,3918
Tipo Organización,4376,23,Sociedades por acciones simplificadas sas,1854
Cámara de Comercio,4377,55,Medellin para antioquia,2005
Fecha de Matrícula,4362,2862,1996/05/01,29
Fecha de Vigencia,4361,1946,Indefinido,2065
Fecha de Cancelación,1,1,2024/12/14,1


### Resultados:

En vista a los resumenes observamos que la mayoría de columnas del dataframe según lo hemos cargado son no númericas, de hecho las dos variables númericas
encontradas son el número de matrícula y el año que no son realmente numéricas.

La mayoría de columnas tienen pocos valores directamente faltantes pero se observan valores que marcan columnas vacías o desconocidas:

- El mínimo del año es 0, marcaremos este dato como nulo
- En identificación el dato más frecuente es la cadena: "SIN IDENTIFICACION"
- En la Fecha de vigencia el dato máß frecuente es la cadena "Indefinido", esto es posible que se de en otros campos de tipo fecha. Sin embargo en estos caso una acción especial no será necesaria: cuando convirtamos el valor a tipo fecha fallará la conversión y tendremos un valor nulo.
- La columna de fecha de cancelación solo tiene un registro no nulo, con lo que esta columna deberá ser completamente descartada.

Otra cosa que podemos observar es varias variables con un número de valores únicos bajo, estas columnas son las propiamente categóricas:

- Estado de la matrícula
- Tipo de sociedad
- Tipo Organización
- Camara de Comercio
- Categoría de matricula
- Motivo de cancelación.

Un par de variables son booleanas marcadas con 'N' o 'S' (O nulos)

- Emprendimiento Social
- Extinción de Dominio

(Estas variables tiene alto número de nulos)

## Limpieza de los datos originales

### Variables categóricas

In [5]:
# Convertir las variables categóricas al tipo adecuado
category_cols = ["Estado de la matrícula", "Tipo de Sociedad", "Tipo Organización", "Cámara de Comercio", "Categoria de la Matrícula"]
df[category_cols] = df[category_cols].astype("category") 

In [7]:
for col in df.select_dtypes(include="category"):
    cats = df[col].cat.categories

    table = (
        pd.DataFrame({"Categoría": cats})
        .to_html(index=False, escape=False)
    )

    display(HTML(f"<h3>{col}</h3>{table}"))

Categoría
Activa
"Activa, constitución por traslado"
Cancelada


Categoría
Establecimiento de comercio
Persona natural
Sociedad ó persona juridica principal ó esal


Categoría
Economia solidaria
Entidad sin animo de lucro
No aplica
Sociedad civil
Sociedad comercial


Categoría
Asociaciones agropecuarias y campesinas nacionales y no nacionales
Asociaciones de padres de familia
"Asociaciones, corporaciones, fundaciones e instituciones de utilidad común (gremiales, de beneficencia; profesionales, juveniles, sociales, democráticas y participativas, cívicas y comunitarias, de egresados, de rehabilitación social y ayuda a indigentes y clubes sociales)."
"Cooperativas, federaciones y confederaciones, instituciones auxiliares de la economía solidaria y precooperativas."
Corporaciones
"Corporaciones, asociaciones y fundaciones creadas para adelantar actividades en comunidades indígenas."
Empresas asociativas de trabajo
Empresas unipersonales
"Entidades científicas, tecnológicas, culturales, e investigativas"
Entidades de naturaleza cooperativa


Categoría
Aburra sur
Aguachica
Amazonas
Arauca
Armenia
Barrancabermeja
Barranquilla
Bogota
Bucaramanga
Buenaventura


# Variables de tipo fecha

In [ ]:
date_cols = ["Fecha de Matrícula", "Fecha de Vigencia", "Fecha de renovación", "Fecha de Actualización"]
df[date_cols] = df[date_cols].apply(pd.to_datetime, format="%Y/%m/%d", errors="coerce")
df

,Identificación,Número de Matrícula,Estado de la matrícula,nombre,Categoria de la Matrícula,Tipo de Sociedad,Tipo Organización,Cámara de Comercio,Fecha de Matrícula,Fecha de Vigencia,Fecha de Cancelación,Último año renovado,Fecha de renovación,Fecha de Actualización,Motivo Cancelación,Emprendimiento Social,Extinción de Dominio
0,NIT 800137336 - 0,15941604,Activa,COMERCIALIZADORA LA LLAMA S A,Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedad anonima,Medellin para antioquia,1991-07-01,2041-02-13,NaN,1992,NaN,NaN,NaN,NaN,NaN
1,NIT 800136312 - 0,15941804,Activa,VELPASS S A,Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedad anonima,Medellin para antioquia,1991-07-01,2021-06-26,NaN,1996,NaN,NaN,NaN,NaN,NaN
2,NIT 800135479 - 6,15942104,Activa,"DILADA S A ""EN LIQUIDACION""",Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedad anonima,Medellin para antioquia,1991-07-01,2041-07-15,NaN,1999,NaN,NaN,NaN,NaN,NaN
3,NIT 800137202 - 2,15968204,Activa,AIRE ACONDICIONADO Y REFRIGERACION DE COLOMBIA...,Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedad anonima,Medellin para antioquia,1991-07-01,2041-06-14,NaN,1998,1998/08/28,NaN,NaN,NaN,NaN
4,NIT 800137881 - 3,15997912,Activa,CASMAR DE COLOMBIA S. A. S.,Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedades por acciones simplificadas sas,Medellin para antioquia,1991-08-01,NaT,NaN,2022,2022/12/20,2022/12/20,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4372,NIT 901285458 - 0,3114804,Activa,EL CONTACTO ELECTRONICO IND COL S.A.S,Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedades por acciones simplificadas sas,Bogota,2019-05-20,NaT,NaN,2025,2025/05/09,2025/05/09,NaN,N,NaN
4373,NIT 901231306 - 8,3037580,Activa,GENESIS CORPORATION IND JANIMAN GMBH S.A.S,Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedades por acciones simplificadas sas,Bogota,2018-11-16,2028-11-15,NaN,2025,2025/05/07,2025/05/07,NaN,N,NaN
4374,NIT 901615150 - 5,844753,Activa,PROCESOS INDUSTRIALES BC S.A.S. Sigla PROC IND...,Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedades por acciones simplificadas sas,Barranquilla,2022-07-21,NaT,NaN,2025,2025/03/19,2025/04/24,NaN,N,NaN
4375,NIT 900834734 - 1,611251,Activa,ESPECIALISTAS EN MANTENIMIENTO Y SOLUCIONES IN...,Sociedad ó persona juridica principal ó esal,Sociedad comercial,Sociedades por acciones simplificadas sas,Barranquilla,2014-11-24,NaT,NaN,2025,2025/03/27,2025/04/24,NaN,N,NaN


### Variables booleanas

In [14]:
bool_cols = ["Emprendimiento Social", "Extinción de Dominio"]
df[bool_cols] = (
    df[bool_cols]
    .apply(lambda s: s.map({"S": True, "N": False}))
    .astype("boolean")
)

### Otros valores nulos

In [15]:
df['Último año renovado'] = df['Último año renovado'].replace(0, pd.NA)

### Columnas inválidas

Descartaremos para el análisis 'Fecha de cancelación' Por el elevado número de nulos. Además descrataremos identificación y número de matrícula por ser columnas identificativas y no características utlizable por un modelo posterior.

In [17]:
df = df.drop(columns = ["Fecha de Cancelación", "Número de Matrícula", "Identificación"])

## Integración de otros datos

Los datos obtenidos mediante web-scraping no tienen características numéricas que podamos utilizar para el posterior análisis.

Por eso es necesario que recuperemos datos de otras fuentes y los crucemos.

(Por ejemplo GDP de colombia y lo cruzamos por año de matrícula ...)

In [20]:
# TODO

# Análisis de datos

In [ ]:

print_header("FASE 3: ANÁLISIS DE DATOS - MODELOS ML Y CONTRASTE DE HIPÓTESIS", 80)


OBJETIVOS_NEGOCIO = {
    'segmentacion_regional': {
        'nombre': 'Segmentación Regional de Empresas',
        'descripcion': 'Identificar patrones geográficos en tipos de sociedad',
        'variables': ['Cámara de Comercio', 'Tipo de Sociedad', 'Estado de la matrícula'],
        'modelo': 'clustering',
        'justificacion': 'Diseñar políticas regionales diferenciadas'
    },
    'prediccion_tipo_sociedad': {
        'nombre': 'Predicción de Tipo de Sociedad por Región',
        'descripcion': 'Predecir el tipo de sociedad más probable por región',
        'variable_objetivo': 'Tipo de Sociedad',
        'variables_predictoras': ['Cámara de Comercio', 'Año_Matricula', 'Estado_Matricula'],
        'modelo': 'clasificacion',
        'justificacion': 'Optimizar procesos de registro y categorización'
    },
    'contraste_geografico': {
        'nombre': 'Contraste Geográfico de Tipos de Sociedad',
        'descripcion': 'Analizar diferencias significativas en distribución de tipos de sociedad por región',
        'hipotesis': 'Existen diferencias significativas en la distribución de tipos de sociedad entre regiones',
        'variables': ['Cámara de Comercio', 'Tipo de Sociedad'],
        'prueba': 'chi-cuadrado',
        'justificacion': 'Validar diferencias regionales para políticas específicas'
    }
}

print_section("1. OBJETIVOS DE ANÁLISIS DEFINIDOS")

for key, objetivo in OBJETIVOS_NEGOCIO.items():
    print(f"\n{Fore.CYAN}{objetivo['nombre']}:")
    print(f"{Fore.WHITE}• Descripción: {objetivo['descripcion']}")
    print(f"{Fore.WHITE}• Justificación: {objetivo['justificacion']}")
    if 'variables' in objetivo:
        print(f"{Fore.WHITE}• Variables: {', '.join(objetivo['variables'])}")


print_section("2. CARGA DEL DATASET LIMPIO")

try:
    rues_df = pd.read_csv('rues_dataset_limpio.csv')
    print(f"{Fore.GREEN} Dataset limpio cargado exitosamente")
    print(f"{Fore.WHITE}• Archivo: rues_dataset_limpio.csv")
    print(f"{Fore.WHITE}• Tamaño: {len(rues_df):,} registros × {len(rues_df.columns):,} columnas")
    
    print(f"\n{Fore.CYAN}Variables disponibles:")
    for i, col in enumerate(rues_df.columns, 1):
        dtype = str(rues_df[col].dtype)
        unique_count = rues_df[col].nunique()
        null_count = rues_df[col].isnull().sum()
        print(f"{i:2}. {col} ({dtype}, {unique_count} únicos, {null_count} nulos)")
        
except FileNotFoundError:
    print(f"{Fore.RED}✗ Error: No se encontró el archivo 'rues_dataset_limpio.csv'")
    print(f"{Fore.YELLOW}• Ejecuta primero la Fase 2: Limpieza de datos")
    raise


print_section("3. PREPROCESAMIENTO PARA ANÁLISIS GEOGRÁFICO")

df_analysis = rues_df.copy()

print_subsection("3.1 Procesamiento de Variables Geográficas")

def extraer_region(camara):
    if pd.isna(camara):
        return "NO ESPECIFICADO"
    
    camara_str = str(camara).upper()
    
    region_mapping = {
        'BOGOTA': 'BOGOTÁ',
        'MEDELLIN': 'ANTIOQUIA',
        'CALI': 'VALLE DEL CAUCA',
        'BARRANQUILLA': 'ATLÁNTICO',
        'CARTAGENA': 'BOLÍVAR',
        'BUCARAMANGA': 'SANTANDER',
        'PEREIRA': 'RISARALDA',
        'MANIZALES': 'CALDAS',
        'IBAGUE': 'TOLIMA',
        'CUCUTA': 'NORTE DE SANTANDER',
        'VILLAVICENCIO': 'META',
        'MONTERIA': 'CÓRDOBA',
        'SANTA MARTA': 'MAGDALENA',
        'ARMENIA': 'QUINDÍO',
        'NEIVA': 'HUILA',
        'PASTO': 'NARIÑO',
        'POPAYAN': 'CAUCA'
    }
    
    for key, region in region_mapping.items():
        if key in camara_str:
            return region
    
    palabras = camara_str.split()
    if len(palabras) > 1:
        if 'DE' in palabras:
            idx = palabras.index('DE')
            if idx + 1 < len(palabras):
                return palabras[idx + 1]
    
    return "OTRA REGIÓN"

df_analysis['Region'] = df_analysis['Cámara de Comercio'].apply(extraer_region)

print(f"{Fore.WHITE}• Regiones identificadas: {df_analysis['Region'].nunique()}")
print(f"{Fore.WHITE}• Distribución por región:")

region_counts = df_analysis['Region'].value_counts().head(10)
for region, count in region_counts.items():
    percentage = (count / len(df_analysis)) * 100
    print(f"  - {region}: {count:,} empresas ({percentage:.1f}%)")

print_subsection("3.2 Procesamiento de Variables Temporales")

def extraer_ano_matricula(fecha_str):
    if pd.isna(fecha_str):
        return np.nan
    
    try:
        fecha = pd.to_datetime(fecha_str, errors='coerce', dayfirst=True)
        if pd.notna(fecha):
            return fecha.year
    except:
        pass
    
    import re
    match = re.search(r'\b(19|20)\d{2}\b', str(fecha_str))
    if match:
        return int(match.group())
    
    return np.nan

df_analysis['Año_Matricula'] = df_analysis['Fecha de Matrícula'].apply(extraer_ano_matricula)

if df_analysis['Año_Matricula'].isnull().sum() > 0:
    median_year = df_analysis['Año_Matricula'].median()
    df_analysis['Año_Matricula'] = df_analysis['Año_Matricula'].fillna(median_year)
    print(f"{Fore.YELLOW}• {df_analysis['Año_Matricula'].isnull().sum()} años nulos imputados con mediana: {median_year:.0f}")

print(f"{Fore.WHITE}• Rango de años de matrícula: {df_analysis['Año_Matricula'].min():.0f} - {df_analysis['Año_Matricula'].max():.0f}")

print_subsection("3.3 Codificación de Variables Categóricas")

from sklearn.preprocessing import LabelEncoder

categorical_vars = ['Region', 'Tipo de Sociedad', 'Estado de la matrícula', 'Categoria de la Matrícula']

label_encoders = {}
for var in categorical_vars:
    if var in df_analysis.columns:
        le = LabelEncoder()
        df_analysis[f'{var}_encoded'] = le.fit_transform(df_analysis[var].astype(str))
        label_encoders[var] = le
        print(f"{Fore.WHITE}• {var}: {df_analysis[var].nunique()} categorías codificadas")


print_section("4. ANÁLISIS DESCRIPTIVO POR REGIÓN")

print_subsection("4.1 Distribución de Tipos de Sociedad por Región")

contingency_table = pd.crosstab(
    df_analysis['Region'], 
    df_analysis['Tipo de Sociedad'],
    margins=True,
    margins_name="TOTAL"
)

print(f"\n{Fore.CYAN}Tabla de contingencia (primeras 10 regiones):")
print(contingency_table.head(11).to_string())

percentage_table = pd.crosstab(
    df_analysis['Region'], 
    df_analysis['Tipo de Sociedad'],
    normalize='index'
) * 100

print(f"\n{Fore.CYAN}Distribución porcentual por región (%):")
top_sociedades = df_analysis['Tipo de Sociedad'].value_counts().head(5).index.tolist()
print(percentage_table[top_sociedades].head(10).round(1).to_string())

print_subsection("4.2 Tipos de Sociedad Más Probables por Región")

region_analysis = []
for region in df_analysis['Region'].unique():
    region_data = df_analysis[df_analysis['Region'] == region]
    
    if len(region_data) > 10:  
        tipo_counts = region_data['Tipo de Sociedad'].value_counts()
        
        if len(tipo_counts) > 0:
            top_tipo = tipo_counts.index[0]
            top_count = tipo_counts.iloc[0]
            top_percentage = (top_count / len(region_data)) * 100
            
            hhi = sum((count / len(region_data))**2 for count in tipo_counts.values) * 10000
            
            region_analysis.append({
                'Region': region,
                'Empresas': len(region_data),
                'Tipo Más Común': top_tipo,
                '% Tipo Más Común': top_percentage,
                'Diversidad (HHI)': hhi,
                'Tipos Únicos': len(tipo_counts)
            })

region_df = pd.DataFrame(region_analysis)
region_df = region_df.sort_values('Empresas', ascending=False)

print(f"\n{Fore.CYAN}Análisis por región:")
headers = ['Region', 'Empresas', 'Tipo Más Común', '% Más Común', 'Diversidad', 'Tipos Únicos']
rows = []
for _, row in region_df.head(15).iterrows():
    rows.append([
        row['Region'],
        f"{row['Empresas']:,}",
        row['Tipo Más Común'][:20],
        f"{row['% Tipo Más Común']:.1f}%",
        f"{row['Diversidad (HHI)']:.0f}",
        row['Tipos Únicos']
    ])

print(f"\n{tabulate(rows, headers=headers, tablefmt='grid')}")

print(f"\n{Fore.MAGENTA}INTERPRETACIÓN INICIAL:")
print(f"{Fore.WHITE}• Regiones con alta concentración (baja diversidad):")
high_concentration = region_df[region_df['Diversidad (HHI)'] > 5000]
for _, row in high_concentration.head(5).iterrows():
    print(f"  - {row['Region']}: {row['Tipo Más Común']} ({row['% Tipo Más Común']:.1f}%)")

print(f"\n{Fore.WHITE}• Regiones con alta diversidad:")
high_diversity = region_df[region_df['Diversidad (HHI)'] < 2000]
for _, row in high_diversity.head(5).iterrows():
    print(f"  - {row['Region']}: {row['Tipos Únicos']} tipos distintos")


print_header("MODELO NO SUPERVISADO: CLUSTERING REGIONAL", 80)

print_section("5.1 Preparación de Datos para Clustering")

from sklearn.preprocessing import StandardScaler

clustering_vars = [
    'Region_encoded',
    'Tipo de Sociedad_encoded',
    'Año_Matricula',
    'Estado de la matrícula_encoded'
]

clustering_vars = [var for var in clustering_vars if var in df_analysis.columns]

X_cluster = df_analysis[clustering_vars].copy()

X_cluster = X_cluster.fillna(X_cluster.median())

print(f"{Fore.WHITE}• Variables para clustering: {len(clustering_vars)}")
print(f"{Fore.WHITE}• Dimensiones: {X_cluster.shape}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"{Fore.WHITE}• Datos estandarizados para clustering")

print_section("5.2 Determinación del Número Óptimo de Clusters")

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

inertias = []
silhouette_scores = []
db_scores = []
k_range = range(2, 11)

print(f"\n{Fore.CYAN}Evaluando número óptimo de clusters...")

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, algorithm='lloyd')
    kmeans.fit(X_scaled)
    
    inertias.append(kmeans.inertia_)
    
    if k > 1:
        labels = kmeans.labels_
        silhouette_avg = silhouette_score(X_scaled, labels)
        silhouette_scores.append(silhouette_avg)
        
        db_score = davies_bouldin_score(X_scaled, labels)
        db_scores.append(db_score)
        
        print(f"{Fore.WHITE}  k={k}: Inercia={kmeans.inertia_:,.0f}, Silhouette={silhouette_avg:.3f}, DB={db_score:.3f}")

if len(silhouette_scores) > 0:
    optimal_k_silhouette = k_range[np.argmax(silhouette_scores)]
else:
    optimal_k_silhouette = 3

if len(db_scores) > 0:
    optimal_k_db = k_range[np.argmin(db_scores)]
else:
    optimal_k_db = 3

optimal_k = max(3, min(optimal_k_silhouette, optimal_k_db, 7))  

print(f"\n{Fore.GREEN} Número óptimo de clusters determinado: {optimal_k}")
print(f"{Fore.WHITE}• Basado en Silhouette: {optimal_k_silhouette}")
print(f"{Fore.WHITE}• Basado en Davies-Bouldin: {optimal_k_db}")

print_section("5.3 Aplicación de K-Means con Clusters Óptimos")

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, algorithm='lloyd')
clusters = kmeans_final.fit_predict(X_scaled)

df_analysis['Cluster'] = clusters

print(f"{Fore.GREEN} Clustering aplicado exitosamente")
print(f"{Fore.WHITE}• Total clusters: {optimal_k}")
print(f"{Fore.WHITE}• Distribución de empresas por cluster:")

cluster_distribution = pd.Series(clusters).value_counts().sort_index()
for cluster_num, count in cluster_distribution.items():
    percentage = (count / len(clusters)) * 100
    print(f"  Cluster {cluster_num}: {count:,} empresas ({percentage:.1f}%)")

print_section("5.4 Análisis e Interpretación de Clusters")

print_subsection("Características por Cluster")

cluster_profiles = []

for cluster_num in range(optimal_k):
    cluster_data = df_analysis[df_analysis['Cluster'] == cluster_num]
    
    if len(cluster_data) > 0:
        top_region = cluster_data['Region'].mode()
        top_region = top_region.iloc[0] if not top_region.empty else "N/A"
        region_pct = (cluster_data['Region'] == top_region).sum() / len(cluster_data) * 100
        
        top_tipo = cluster_data['Tipo de Sociedad'].mode()
        top_tipo = top_tipo.iloc[0] if not top_tipo.empty else "N/A"
        tipo_pct = (cluster_data['Tipo de Sociedad'] == top_tipo).sum() / len(cluster_data) * 100
        
        avg_year = cluster_data['Año_Matricula'].mean()
        
        top_estado = cluster_data['Estado de la matrícula'].mode()
        top_estado = top_estado.iloc[0] if not top_estado.empty else "N/A"
        
        cluster_profiles.append({
            'Cluster': cluster_num,
            'Empresas': len(cluster_data),
            '% Total': (len(cluster_data) / len(df_analysis)) * 100,
            'Región Predominante': top_region,
            '% Región': region_pct,
            'Tipo Predominante': top_tipo,
            '% Tipo': tipo_pct,
            'Año Promedio': avg_year,
            'Estado Predominante': top_estado
        })

profiles_df = pd.DataFrame(cluster_profiles)
print(f"\n{Fore.CYAN}Perfiles de clusters identificados:")
print(profiles_df.to_string())

print_subsection("Interpretación de Segmentos")

print(f"\n{Fore.MAGENTA}INTERPRETACIÓN DE CLUSTERS:")

for profile in cluster_profiles:
    print(f"\n{Fore.CYAN}Cluster {profile['Cluster']} ({profile['Empresas']:,} empresas, {profile['% Total']:.1f}%):")
    print(f"{Fore.WHITE}• Región característica: {profile['Región Predominante']} ({profile['% Región']:.1f}%)")
    print(f"{Fore.WHITE}• Tipo de sociedad típico: {profile['Tipo Predominante']} ({profile['% Tipo']:.1f}%)")
    print(f"{Fore.WHITE}• Año promedio de matrícula: {profile['Año Promedio']:.0f}")
    print(f"{Fore.WHITE}• Estado predominante: {profile['Estado Predominante']}")
    
    if profile['% Tipo'] > 60:
        print(f"{Fore.YELLOW}  Segmento especializado: Políticas específicas para {profile['Tipo Predominante']}")
    elif profile['% Región'] > 70:
        print(f"{Fore.YELLOW}   Segmento regional: Enfoque geográfico en {profile['Región Predominante']}")
    elif profile['Año Promedio'] > 2015:
        print(f"{Fore.YELLOW}   Segmento de empresas jóvenes: Programas de apoyo a emprendimientos")
    else:
        print(f"{Fore.YELLOW}   Segmento diversificado: Enfoque generalista")


print_header("MODELO SUPERVISADO: PREDICCIÓN DE TIPO DE SOCIEDAD", 80)

print_section("6.1 Preparación de Datos para Clasificación")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

top_tipos = df_analysis['Tipo de Sociedad'].value_counts().head(5).index.tolist()
df_classification = df_analysis[df_analysis['Tipo de Sociedad'].isin(top_tipos)].copy()

print(f"{Fore.WHITE}• Tipos de sociedad a predecir (top 5): {', '.join(top_tipos)}")
print(f"{Fore.WHITE}• Empresas para clasificación: {len(df_classification):,}")

feature_vars = [
    'Region_encoded',
    'Año_Matricula',
    'Estado de la matrícula_encoded',
    'Categoria de la Matrícula_encoded'
]

feature_vars = [var for var in feature_vars if var in df_classification.columns]

X = df_classification[feature_vars].copy()
y = df_classification['Tipo de Sociedad'].copy()

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print(f"{Fore.WHITE}• Variables predictoras: {len(feature_vars)}")
print(f"{Fore.WHITE}• Clases a predecir: {len(np.unique(y_encoded))}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.3, 
    random_state=42,
    stratify=y_encoded
)

print(f"{Fore.WHITE}• Conjuntos de datos:")
print(f"  - Entrenamiento: {X_train.shape[0]:,} muestras")
print(f"  - Prueba: {X_test.shape[0]:,} muestras")

print_section("6.2 Entrenamiento del Modelo Random Forest")

rf_model = RandomForestClassifier(
    n_estimators=100,  
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=1,  
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

print(f"{Fore.GREEN} Modelo Random Forest entrenado")
print(f"{Fore.WHITE}• Hiperparámetros:")
print(f"  - Número de árboles: {rf_model.n_estimators}")
print(f"  - Profundidad máxima: {rf_model.max_depth}")
print(f"  - Muestras mínimas para dividir: {rf_model.min_samples_split}")

print_section("6.3 Evaluación del Modelo")

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n{Fore.CYAN}Métricas de evaluación:")
print(f"{Fore.WHITE}• Accuracy en test: {accuracy:.3f}")

try:
    cv_scores = cross_val_score(rf_model, X, y_encoded, cv=3, scoring='accuracy', n_jobs=1)
    print(f"{Fore.WHITE}• Accuracy (Cross-Validation 3-fold): {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
except:
    print(f"{Fore.YELLOW}• Validación cruzada omitida debido a limitaciones de memoria")

print(f"\n{Fore.CYAN}Reporte de clasificación:")
target_names = le_target.classes_
print(classification_report(y_test, y_pred, target_names=target_names))

print(f"\n{Fore.CYAN}Matriz de confusión (normalizada - diagonal principal):")
conf_matrix = confusion_matrix(y_test, y_pred, normalize='true')
for i, class_name in enumerate(target_names):
    accuracy_class = conf_matrix[i, i]
    print(f"  • {class_name}: {accuracy_class:.3f}")

print_section("6.4 Importancia de Características")

feature_importance = pd.DataFrame({
    'Variable': feature_vars,
    'Importancia': rf_model.feature_importances_
}).sort_values('Importancia', ascending=False)

print(f"\n{Fore.CYAN}Importancia de características:")
print(feature_importance.to_string())

print(f"\n{Fore.MAGENTA}INTERPRETACIÓN DE PREDICTORES:")
for _, row in feature_importance.iterrows():
    var = row['Variable']
    importance = row['Importancia'] * 100
    
    if 'Region' in var:
        print(f"{Fore.WHITE}• Región geográfica: {importance:.1f}% de importancia")
        print(f"   La ubicación es el predictor más fuerte del tipo de sociedad")
    elif 'Año' in var:
        print(f"{Fore.WHITE}• Año de matrícula: {importance:.1f}% de importancia")
        print(f"   La temporalidad influye en el tipo de sociedad elegido")
    elif 'Estado' in var:
        print(f"{Fore.WHITE}• Estado de matrícula: {importance:.1f}% de importancia")
        print(f"   El estado actual se correlaciona con el tipo de sociedad")
    elif 'Categoria' in var:
        print(f"{Fore.WHITE}• Categoría de matrícula: {importance:.1f}% de importancia")
        print(f"   La categoría está relacionada con el tipo de sociedad")

print_section("6.5 Predicción por Región")

region_predictions = []

for region in df_analysis['Region'].unique():
    region_data = df_analysis[df_analysis['Region'] == region]
    
    if len(region_data) > 50:  
        region_features = region_data[feature_vars].copy()
        
        for col in region_features.columns:
            if region_features[col].isnull().any():
                region_features[col] = region_features[col].fillna(X[col].median())
        
        if not region_features.empty:
            try:
                probas = rf_model.predict_proba(region_features)
                
                avg_probas = probas.mean(axis=0)
                
                most_probable_idx = np.argmax(avg_probas)
                most_probable_type = le_target.inverse_transform([most_probable_idx])[0]
                probability = avg_probas[most_probable_idx] * 100
                
                region_predictions.append({
                    'Region': region,
                    'Empresas Muestra': len(region_data),
                    'Tipo Más Probable': most_probable_type,
                    'Probabilidad': probability,
                    'Segundo Más Probable': le_target.inverse_transform([np.argsort(avg_probas)[-2]])[0],
                    'Probabilidad 2do': avg_probas[np.argsort(avg_probas)[-2]] * 100
                })
            except Exception as e:
                continue

if region_predictions:
    pred_df = pd.DataFrame(region_predictions)
    pred_df = pred_df.sort_values('Probabilidad', ascending=False)

    print(f"\n{Fore.CYAN}Predicción de tipo de sociedad más probable por región (top 5):")
    headers = ['Region', 'Empresas', 'Tipo Más Probable', 'Probabilidad', '2do Más Probable', 'Prob 2do']
    rows = []
    for _, row in pred_df.head(5).iterrows():
        rows.append([
            row['Region'],
            f"{row['Empresas Muestra']:,}",
            row['Tipo Más Probable'][:15],
            f"{row['Probabilidad']:.1f}%",
            row['Segundo Más Probable'][:15],
            f"{row['Probabilidad 2do']:.1f}%"
        ])

    print(f"\n{tabulate(rows, headers=headers, tablefmt='grid')}")
else:
    print(f"\n{Fore.YELLOW} No se pudieron generar predicciones por región")


print_header("CONTRASTE DE HIPÓTESIS ESTADÍSTICO", 80)

print_section("7.1 Formulación de Hipótesis")

print(f"\n{Fore.CYAN}Hipótesis a contrastar:")
print(f"{Fore.WHITE}• H₀ (Hipótesis nula): No existen diferencias significativas en la distribución")
print(f"{Fore.WHITE}  de tipos de sociedad entre regiones geográficas")
print(f"{Fore.WHITE}• H₁ (Hipótesis alternativa): Existen diferencias significativas en la distribución")
print(f"{Fore.WHITE}  de tipos de sociedad entre regiones geográficas")

print(f"\n{Fore.CYAN}Variables analizadas:")
print(f"{Fore.WHITE}• Variable independiente: Región geográfica")
print(f"{Fore.WHITE}• Variable dependiente: Tipo de Sociedad")
print(f"{Fore.WHITE}• Prueba seleccionada: Chi-cuadrado de independencia")

print_section("7.2 Preparación de Datos para Prueba Chi-Cuadrado")

from scipy.stats import chi2_contingency

top_regions = df_analysis['Region'].value_counts().head(8).index.tolist()  
top_tipos = df_analysis['Tipo de Sociedad'].value_counts().head(4).index.tolist()  

filtered_data = df_analysis[
    df_analysis['Region'].isin(top_regions) & 
    df_analysis['Tipo de Sociedad'].isin(top_tipos)
].copy()

print(f"{Fore.WHITE}• Regiones analizadas: {len(top_regions)} principales")
print(f"{Fore.WHITE}• Tipos de sociedad analizados: {len(top_tipos)} principales")
print(f"{Fore.WHITE}• Muestra para prueba: {len(filtered_data):,} empresas")

contingency_table_chi = pd.crosstab(
    filtered_data['Region'],
    filtered_data['Tipo de Sociedad']
)

print(f"\n{Fore.CYAN}Tabla de contingencia para prueba chi-cuadrado:")
print(contingency_table_chi.to_string())

print_section("7.3 Verificación de Supuestos")

print(f"\n{Fore.CYAN}Verificando supuestos para prueba chi-cuadrado:")

print(f"{Fore.WHITE} Ambas variables son categóricas: REGIÓN y TIPO DE SOCIEDAD")

print(f"{Fore.WHITE} Se asume independencia: cada empresa aparece solo una vez")

total_observations = contingency_table_chi.sum().sum()
print(f"{Fore.WHITE}• Total observaciones: {total_observations:,}")

chi2, p, dof, expected = chi2_contingency(contingency_table_chi)

expected_flat = expected.flatten()
cells_above_5 = sum(expected_flat > 5)
total_cells = len(expected_flat)
percentage_above_5 = (cells_above_5 / total_cells) * 100

print(f"{Fore.WHITE}• Celdas con frecuencia esperada > 5: {cells_above_5}/{total_cells} ({percentage_above_5:.1f}%)")

if percentage_above_5 >= 80:
    print(f"{Fore.GREEN} Supuesto cumplido: {percentage_above_5:.1f}% de celdas con frecuencia > 5")
else:
    print(f"{Fore.YELLOW} Advertencia: Solo {percentage_above_5:.1f}% de celdas con frecuencia > 5")
    print(f"{Fore.WHITE}  Considerar agrupar categorías o usar prueba exacta de Fisher")

print_section("7.4 Aplicación de Prueba Chi-Cuadrado")

print(f"\n{Fore.CYAN}Aplicando prueba chi-cuadrado de independencia...")

chi2, p_value, dof, expected = chi2_contingency(contingency_table_chi)

print(f"{Fore.WHITE}• Estadístico chi-cuadrado: χ² = {chi2:.2f}")
print(f"{Fore.WHITE}• Grados de libertad: df = {dof}")
print(f"{Fore.WHITE}• p-valor: p = {p_value:.6f}")

alpha = 0.05
print(f"{Fore.WHITE}• Nivel de significancia: α = {alpha}")

print_section("7.5 Interpretación de Resultados")

print(f"\n{Fore.CYAN}Decisión estadística:")

if p_value < alpha:
    print(f"{Fore.RED}• RECHAZAMOS la hipótesis nula H₀ (p < {alpha})")
    print(f"{Fore.WHITE}   Existen diferencias estadísticamente significativas")
    print(f"{Fore.WHITE}   La distribución de tipos de sociedad NO es independiente de la región")
else:
    print(f"{Fore.GREEN}• NO RECHAZAMOS la hipótesis nula H₀ (p ≥ {alpha})")
    print(f"{Fore.WHITE}   No hay evidencia suficiente de diferencias significativas")
    print(f"{Fore.WHITE}   La distribución de tipos de sociedad parece independiente de la región")

print(f"\n{Fore.CYAN}Medidas de asociación:")

def cramers_v(contingency_table):
    chi2 = chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    return np.sqrt(chi2 / (n * min_dim))

cramers_v_value = cramers_v(contingency_table_chi)
print(f"{Fore.WHITE}• V de Cramer: {cramers_v_value:.3f}")

if cramers_v_value < 0.1:
    strength = "Muy débil"
elif cramers_v_value < 0.3:
    strength = "Débil"
elif cramers_v_value < 0.5:
    strength = "Moderada"
else:
    strength = "Fuerte"

print(f"{Fore.WHITE}• Fuerza de asociación: {strength}")


print_header("SÍNTESIS Y RECOMENDACIONES FINALES", 80)

print_section("8.1 Hallazgos Principales")

print(f"\n{Fore.MAGENTA}1. SEGMENTACIÓN REGIONAL (Clustering):")
print(f"{Fore.WHITE}• Se identificaron {optimal_k} segmentos naturales de empresas")
print(f"{Fore.WHITE}• Cada cluster tiene un perfil regional y de tipo de sociedad distintivo")

print(f"\n{Fore.MAGENTA}2. PREDICCIÓN DE TIPO DE SOCIEDAD (Clasificación):")
print(f"{Fore.WHITE}• Accuracy del modelo: {accuracy:.3f}")
if feature_importance.shape[0] > 0:
    print(f"{Fore.WHITE}• Predictor más importante: {feature_importance.iloc[0]['Variable']}")

print(f"\n{Fore.MAGENTA}3. CONTRASTE DE HIPÓTESIS:")
print(f"{Fore.WHITE}• Resultado: {'DIFERENCIAS SIGNIFICATIVAS' if p_value < alpha else 'NO HAY DIFERENCIAS SIGNIFICATIVAS'}")
print(f"{Fore.WHITE}• Fuerza de asociación: {strength} (V de Cramer: {cramers_v_value:.3f})")


print_section("9. GUARDADO DE RESULTADOS")

output_file = 'rues_analisis_completo.csv'
df_analysis.to_csv(output_file, index=False, encoding='utf-8')
print(f"\n{Fore.GREEN} Dataset completo con análisis guardado: {output_file}")

clusters_file = 'resultados_clustering.csv'
profiles_df.to_csv(clusters_file, index=False, encoding='utf-8')
print(f"{Fore.GREEN} Resultados de clustering: {clusters_file}")

importance_file = 'importancia_caracteristicas.csv'
feature_importance.to_csv(importance_file, index=False, encoding='utf-8')
print(f"{Fore.GREEN} Importancia de características: {importance_file}")

if 'pred_df' in locals() and not pred_df.empty:
    predictions_file = 'predicciones_por_region.csv'
    pred_df.to_csv(predictions_file, index=False, encoding='utf-8')
    print(f"{Fore.GREEN} Predicciones por región: {predictions_file}")

chi2_results = {
    'chi2_statistic': chi2,
    'p_value': p_value,
    'degrees_freedom': dof,
    'cramers_v': cramers_v_value,
    'significance': p_value < alpha,
    'strength_of_association': strength
}

chi2_df = pd.DataFrame([chi2_results])
chi2_file = 'resultados_chi_cuadrado.csv'
chi2_df.to_csv(chi2_file, index=False, encoding='utf-8')
print(f"{Fore.GREEN} Resultados chi-cuadrado: {chi2_file}")

report_file = 'reporte_analisis_completo.txt'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write("REPORTE DE ANÁLISIS COMPLETO - DATASET RUES\n")
    f.write("=" * 70 + "\n\n")
    
    f.write("1. RESUMEN EJECUTIVO:\n")
    f.write(f"   • Fecha de análisis: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"   • Empresas analizadas: {len(df_analysis):,}\n")
    f.write(f"   • Regiones identificadas: {df_analysis['Region'].nunique()}\n")
    f.write(f"   • Tipos de sociedad: {df_analysis['Tipo de Sociedad'].nunique()}\n\n")
    
    f.write("2. MODELO NO SUPERVISADO (CLUSTERING):\n")
    f.write(f"   • Clusters identificados: {optimal_k}\n")
    f.write(f"   • Distribución por cluster:\n")
    for profile in cluster_profiles:
        f.write(f"     - Cluster {profile['Cluster']}: {profile['Empresas']:,} empresas ({profile['% Total']:.1f}%)\n")
    
    f.write("\n3. MODELO SUPERVISADO (CLASIFICACIÓN):\n")
    f.write(f"   • Accuracy del modelo: {accuracy:.3f}\n")
    
    f.write("\n4. CONTRASTE DE HIPÓTESIS:\n")
    f.write(f"   • p-valor: {p_value:.6f}\n")
    f.write(f"   • Decisión: {'DIFERENCIAS SIGNIFICATIVAS' if p_value < alpha else 'NO HAY DIFERENCIAS SIGNIFICATIVAS'}\n")
    f.write(f"   • V de Cramer: {cramers_v_value:.3f} ({strength})\n")
    
        
    f.write("\n6. ARCHIVOS GENERADOS:\n")
    f.write(f"   • Dataset completo: {output_file}\n")
    f.write(f"   • Resultados clustering: {clusters_file}\n")
    f.write(f"   • Importancia características: {importance_file}\n")
    f.write(f"   • Resultados chi-cuadrado: {chi2_file}\n")

print(f"{Fore.GREEN} Reporte ejecutivo: {report_file}")

print(f"\n{Fore.LIGHTBLACK_EX}{'='*80}")
print(f"{Fore.GREEN}{'ANÁLISIS COMPLETADO EXITOSAMENTE'.center(80)}")
print(f"{Fore.LIGHTBLACK_EX}{'='*80}")

rues_df_analizado = df_analysis


NameError: name 'init' is not defined

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

init(autoreset=True)

def print_header(title, width=80):
    print(f"\n{Fore.CYAN}{'=' * width}")
    print(f"{Fore.YELLOW}{Style.BRIGHT}{title.center(width)}")
    print(f"{Fore.CYAN}{'=' * width}{Style.RESET_ALL}")

def print_section(title):
    print(f"\n{Fore.GREEN}{Style.BRIGHT}{title}")
    print(f"{Fore.GREEN}{'-' * 60}{Style.RESET_ALL}")

def print_subsection(title):
    print(f"\n{Fore.MAGENTA}{title}{Style.RESET_ALL}")

print_header("FASE 4: REPRESENTACIÓN VISUAL DE RESULTADOS", 80)


print_section("1. CARGA DE DATOS ANALIZADOS")

try:
    df_analysis = pd.read_csv('rues_analisis_completo.csv')
    print(f"{Fore.GREEN} Dataset analizado cargado exitosamente")
    print(f"{Fore.WHITE}• Archivo: rues_analisis_completo.csv")
    print(f"{Fore.WHITE}• Tamaño: {len(df_analysis):,} registros × {len(df_analysis.columns):,} columnas")
    
    try:
        clusters_df = pd.read_csv('resultados_clustering.csv')
        print(f"{Fore.GREEN} Resultados de clustering cargados")
    except:
        print(f"{Fore.YELLOW} Resultados de clustering no encontrados")
        clusters_df = None
        
    try:
        importance_df = pd.read_csv('importancia_caracteristicas.csv')
        print(f"{Fore.GREEN} Importancia de características cargada")
    except:
        print(f"{Fore.YELLOW} Importancia de características no encontrada")
        importance_df = None
        
except FileNotFoundError:
    print(f"{Fore.RED}✗ Error: No se encontró el archivo 'rues_analisis_completo.csv'")
    print(f"{Fore.YELLOW}• Ejecuta primero la Fase 3: Análisis de datos")
    raise


print_header("VISUALIZACIÓN DEL DATASET LIMPIO", 80)

print_section("2.1 Distribución de Variables Principales")

fig1, axes1 = plt.subplots(2, 3, figsize=(18, 12))
fig1.suptitle('Distribución de Variables Principales del Dataset RUES', fontsize=16, fontweight='bold')

ax1 = axes1[0, 0]
if 'Region' in df_analysis.columns:
    top_regions = df_analysis['Region'].value_counts().head(10)
    colors1 = cm.viridis(np.linspace(0.2, 0.8, len(top_regions)))
    bars1 = ax1.barh(top_regions.index, top_regions.values, color=colors1)
    ax1.set_xlabel('Número de Empresas')
    ax1.set_title('Top 10 Regiones por Número de Empresas')
    ax1.invert_yaxis() 
    
    for i, (bar, value) in enumerate(zip(bars1, top_regions.values)):
        ax1.text(value + max(top_regions.values)*0.01, bar.get_y() + bar.get_height()/2, 
                f'{value:,}', va='center', fontsize=9)

ax2 = axes1[0, 1]
if 'Tipo de Sociedad' in df_analysis.columns:
    top_sociedades = df_analysis['Tipo de Sociedad'].value_counts().head(10)
    colors2 = cm.plasma(np.linspace(0.2, 0.8, len(top_sociedades)))
    
    pie_result = ax2.pie(top_sociedades.values, labels=None, 
                        colors=colors2, autopct='%1.1f%%',
                        startangle=90, counterclock=False)
    
    if len(pie_result) == 3:
        wedges2, texts2, autotexts2 = pie_result
    elif len(pie_result) == 2:
        wedges2, texts2 = pie_result
        autotexts2 = []
    else:
        wedges2 = pie_result[0]
        texts2 = []
        autotexts2 = []
    
    ax2.set_title('Distribución de Tipos de Sociedad (Top 10)')
    
    ax2.legend(wedges2, top_sociedades.index, title="Tipos", 
              loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

ax3 = axes1[0, 2]
if 'Estado de la matrícula' in df_analysis.columns:
    estados = df_analysis['Estado de la matrícula'].value_counts()
    colors3 = cm.Set3(np.linspace(0, 1, len(estados)))
    bars3 = ax3.bar(range(len(estados)), estados.values, color=colors3)
    ax3.set_xlabel('Estado de Matrícula')
    ax3.set_ylabel('Número de Empresas')
    ax3.set_title('Distribución por Estado de Matrícula')
    ax3.set_xticks(range(len(estados)))
    ax3.set_xticklabels(estados.index, rotation=45, ha='right')
    
    for bar, value in zip(bars3, estados.values):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + max(estados.values)*0.01,
                f'{value:,}', ha='center', va='bottom', fontsize=9)

ax4 = axes1[1, 0]
if 'Año_Matricula' in df_analysis.columns:
    years_clean = df_analysis['Año_Matricula'].dropna()
    years_clean = years_clean[(years_clean >= 1900) & (years_clean <= 2024)]
    
    ax4.hist(years_clean, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    ax4.set_xlabel('Año de Matrícula')
    ax4.set_ylabel('Frecuencia')
    ax4.set_title('Distribución de Años de Matrícula')
    ax4.grid(True, alpha=0.3)
    
    mean_year = years_clean.mean()
    median_year = years_clean.median()
    ax4.axvline(mean_year, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_year:.0f}')
    ax4.axvline(median_year, color='green', linestyle='--', linewidth=2, label=f'Mediana: {median_year:.0f}')
    ax4.legend()

ax5 = axes1[1, 1]
numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 1:
    corr_matrix = df_analysis[numeric_cols].corr()
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=.5, cbar_kws={"shrink": .8}, ax=ax5)
    ax5.set_title('Matriz de Correlación de Variables Numéricas')
    
    ax5.set_xticklabels(ax5.get_xticklabels(), rotation=45, ha='right')
    ax5.set_yticklabels(ax5.get_yticklabels(), rotation=0)
else:
    ax5.text(0.5, 0.5, 'No hay suficientes variables\nnuméricas para correlación', 
             ha='center', va='center', fontsize=12)
    ax5.set_title('Matriz de Correlación')

ax6 = axes1[1, 2]
if 'Region' in df_analysis.columns and 'Año_Matricula' in df_analysis.columns:
    top_5_regions = df_analysis['Region'].value_counts().head(5).index.tolist()
    df_top_regions = df_analysis[df_analysis['Region'].isin(top_5_regions)]
    
    box_data = [df_top_regions[df_top_regions['Region'] == region]['Año_Matricula'].dropna() 
                for region in top_5_regions]
    
    bp = ax6.boxplot(box_data, labels=top_5_regions, patch_artist=True)
    
    colors_box = cm.Accent(np.linspace(0, 1, len(top_5_regions)))
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
    
    ax6.set_xlabel('Región')
    ax6.set_ylabel('Año de Matrícula')
    ax6.set_title('Distribución de Años de Matrícula por Región (Top 5)')
    ax6.tick_params(axis='x', rotation=45)
    ax6.grid(True, alpha=0.3)
else:
    ax6.text(0.5, 0.5, 'Datos insuficientes\npara boxplot', 
             ha='center', va='center', fontsize=12)
    ax6.set_title('Boxplot por Región')

plt.tight_layout()
plt.savefig('distribucion_variables_principales.png', dpi=300, bbox_inches='tight')
print(f"{Fore.GREEN} Gráfico 1 guardado: distribucion_variables_principales.png")
plt.show()

print_section("2.2 Análisis Geográfico Detallado")

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 8))
fig2.suptitle('Análisis Geográfico de Empresas por Región', fontsize=16, fontweight='bold')

ax7 = axes2[0]
if 'Region' in df_analysis.columns and 'Tipo de Sociedad' in df_analysis.columns:
    region_stats = []
    for region in df_analysis['Region'].unique():
        region_data = df_analysis[df_analysis['Region'] == region]
        
        if len(region_data) > 10:  
            num_empresas = len(region_data)
            
            tipos_unicos = region_data['Tipo de Sociedad'].nunique()
            
            tipo_mas_comun_pct = region_data['Tipo de Sociedad'].value_counts().iloc[0] / num_empresas * 100
            
            region_stats.append({
                'Region': region,
                'Empresas': num_empresas,
                'Diversidad': tipos_unicos,
                'Concentracion': tipo_mas_comun_pct
            })
    
    if region_stats:
        stats_df = pd.DataFrame(region_stats)
        
        scatter = ax7.scatter(stats_df['Empresas'], stats_df['Diversidad'], 
                            c=stats_df['Concentracion'], 
                            s=stats_df['Empresas']/100,  
                            cmap='viridis', alpha=0.7, edgecolors='black', linewidth=0.5)
        
        ax7.set_xlabel('Número de Empresas (log scale)')
        ax7.set_ylabel('Diversidad (Tipos Únicos)')
        ax7.set_title('Empresas vs Diversidad por Región')
        ax7.set_xscale('log')
        ax7.grid(True, alpha=0.3)
        
        cbar = plt.colorbar(scatter, ax=ax7)
        cbar.set_label('Concentración del Tipo Más Común (%)')
        
        for idx, row in stats_df.nlargest(3, 'Empresas').iterrows():
            ax7.annotate(row['Region'], 
                        (row['Empresas'], row['Diversidad']),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=9, fontweight='bold')
    else:
        ax7.text(0.5, 0.5, 'No hay suficientes datos\npara análisis regional', 
                 ha='center', va='center', fontsize=12)
        ax7.set_title('Empresas vs Diversidad')
else:
    ax7.text(0.5, 0.5, 'Variables necesarias\nno disponibles', 
             ha='center', va='center', fontsize=12)
    ax7.set_title('Análisis Regional')

ax8 = axes2[1]
if 'Region' in df_analysis.columns and 'Tipo de Sociedad' in df_analysis.columns:
    top_5_regions = df_analysis['Region'].value_counts().head(5).index.tolist()
    top_5_tipos = df_analysis['Tipo de Sociedad'].value_counts().head(5).index.tolist()
    
    composition_data = []
    for region in top_5_regions:
        region_data = df_analysis[df_analysis['Region'] == region]
        region_total = len(region_data)
        
        row = {'Region': region}
        for tipo in top_5_tipos:
            count = len(region_data[region_data['Tipo de Sociedad'] == tipo])
            row[tipo] = (count / region_total * 100) if region_total > 0 else 0
        
        composition_data.append(row)
    
    comp_df = pd.DataFrame(composition_data)
    comp_df.set_index('Region', inplace=True)
    
    bottom = np.zeros(len(top_5_regions))
    colors_comp = cm.tab20c(np.linspace(0, 1, len(top_5_tipos)))
    
    for i, tipo in enumerate(top_5_tipos):
        values = comp_df[tipo].values
        ax8.barh(top_5_regions, values, left=bottom, 
                color=colors_comp[i], edgecolor='white', label=tipo[:20])
        bottom += values
    
    ax8.set_xlabel('Porcentaje (%)')
    ax8.set_title('Composición de Tipos de Sociedad por Región (Top 5)')
    ax8.legend(title='Tipo de Sociedad', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax8.invert_yaxis()  
    ax8.grid(True, alpha=0.3, axis='x')
else:
    ax8.text(0.5, 0.5, 'Variables necesarias\nno disponibles', 
             ha='center', va='center', fontsize=12)
    ax8.set_title('Composición por Región')

plt.tight_layout()
plt.savefig('analisis_geografico_detallado.png', dpi=300, bbox_inches='tight')
print(f"{Fore.GREEN} Gráfico 2 guardado: analisis_geografico_detallado.png")
plt.show()


print_header("VISUALIZACIÓN DE RESULTADOS DEL CLUSTERING", 80)

print_section("3.1 Representación de Clusters")

fig3 = plt.figure(figsize=(18, 10))
fig3.suptitle('Análisis de Segmentación por Clustering', fontsize=16, fontweight='bold')

gs = fig3.add_gridspec(2, 3)

ax9 = fig3.add_subplot(gs[0, 0])
if 'Cluster' in df_analysis.columns:
    cluster_counts = df_analysis['Cluster'].value_counts().sort_index()
    colors_cluster = cm.Set2(np.linspace(0, 1, len(cluster_counts)))
    
    pie_result = ax9.pie(cluster_counts.values, labels=None, 
                        colors=colors_cluster, autopct='%1.1f%%',
                        startangle=90, counterclock=False,
                        wedgeprops=dict(edgecolor='w', linewidth=2))
    
    if len(pie_result) == 3:
        wedges, texts, autotexts = pie_result
    else:
        wedges = pie_result[0]
        texts = []
        autotexts = []
    
    ax9.set_title(f'Distribución de Empresas por Cluster\n(Total Clusters: {len(cluster_counts)})')
    
    legend_labels = []
    for cluster, count in cluster_counts.items():
        pct = (count / len(df_analysis)) * 100
        legend_labels.append(f'Cluster {cluster}: {count:,} empresas ({pct:.1f}%)')
    
    ax9.legend(wedges, legend_labels, title="Clusters", 
              loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
else:
    ax9.text(0.5, 0.5, 'No hay datos\nde clusters', 
             ha='center', va='center', fontsize=12)
    ax9.set_title('Distribución de Clusters')

ax10 = fig3.add_subplot(gs[0, 1])
if 'Cluster' in df_analysis.columns and 'Region' in df_analysis.columns:
    cluster_regions = []
    for cluster in sorted(df_analysis['Cluster'].unique()):
        cluster_data = df_analysis[df_analysis['Cluster'] == cluster]
        
        if len(cluster_data) > 0:
            top_region = cluster_data['Region'].mode()
            top_region = top_region.iloc[0] if not top_region.empty else "N/A"
            region_pct = (cluster_data['Region'] == top_region).sum() / len(cluster_data) * 100
            
            top_tipo = cluster_data['Tipo de Sociedad'].mode()
            top_tipo = top_tipo.iloc[0] if not top_tipo.empty else "N/A"
            
            cluster_regions.append({
                'Cluster': cluster,
                'Empresas': len(cluster_data),
                'Región Predominante': top_region,
                '% Región': region_pct,
                'Tipo Predominante': top_tipo
            })
    
    if cluster_regions:
        clusters_reg_df = pd.DataFrame(cluster_regions)
        
        x = np.arange(len(clusters_reg_df))
        width = 0.35
        
        bars1 = ax10.bar(x - width/2, clusters_reg_df['Empresas'], width, 
                        label='Número de Empresas', color='steelblue', alpha=0.7)
        bars2 = ax10.bar(x + width/2, clusters_reg_df['% Región'], width, 
                        label='% Región Predominante', color='coral', alpha=0.7)
        
        ax10.set_xlabel('Cluster')
        ax10.set_ylabel('Valores')
        ax10.set_title('Composición Regional por Cluster')
        ax10.set_xticks(x)
        ax10.set_xticklabels([f'C{c}' for c in clusters_reg_df['Cluster']])
        ax10.legend()
        ax10.grid(True, alpha=0.3, axis='y')
        
        for idx, row in clusters_reg_df.iterrows():
            ax10.text(idx, max(row['Empresas'], row['% Región']) + 5, 
                     row['Región Predominante'][:15], 
                     ha='center', va='bottom', fontsize=8, rotation=45)
    else:
        ax10.text(0.5, 0.5, 'No hay datos\npara composición', 
                 ha='center', va='center', fontsize=12)
        ax10.set_title('Composición por Cluster')
else:
    ax10.text(0.5, 0.5, 'Variables necesarias\nno disponibles', 
             ha='center', va='center', fontsize=12)
    ax10.set_title('Composición Regional')

ax11 = fig3.add_subplot(gs[0, 2])
if 'Cluster' in df_analysis.columns:
    compare_vars = ['Año_Matricula', 'Region_encoded', 'Tipo de Sociedad_encoded']
    compare_vars = [var for var in compare_vars if var in df_analysis.columns]
    
    if len(compare_vars) >= 2:
        cluster_comparison = []
        for cluster in sorted(df_analysis['Cluster'].unique()):
            cluster_data = df_analysis[df_analysis['Cluster'] == cluster]
            
            if len(cluster_data) > 0:
                means = {'Cluster': cluster, 'Empresas': len(cluster_data)}
                for var in compare_vars[:2]:  
                    if var in cluster_data.columns:
                        means[var] = cluster_data[var].mean()
                
                cluster_comparison.append(means)
        
        if cluster_comparison:
            comp_df = pd.DataFrame(cluster_comparison)
            
            scatter = ax11.scatter(comp_df[compare_vars[0]], 
                                  comp_df[compare_vars[1]], 
                                  s=comp_df['Empresas']/10,  
                                  c=comp_df['Cluster'], 
                                  cmap='viridis', alpha=0.7, edgecolors='black')
            
            ax11.set_xlabel(compare_vars[0].replace('_', ' ').title())
            ax11.set_ylabel(compare_vars[1].replace('_', ' ').title())
            ax11.set_title('Comparación de Clusters por Características')
            ax11.grid(True, alpha=0.3)
            
            for idx, row in comp_df.iterrows():
                ax11.annotate(f"C{row['Cluster']}", 
                            (row[compare_vars[0]], row[compare_vars[1]]),
                            xytext=(5, 5), textcoords='offset points',
                            fontsize=9)
        else:
            ax11.text(0.5, 0.5, 'No hay datos\npara comparación', 
                     ha='center', va='center', fontsize=12)
            ax11.set_title('Comparación de Clusters')
    else:
        ax11.text(0.5, 0.5, 'Variables insuficientes\npara comparación', 
                 ha='center', va='center', fontsize=12)
        ax11.set_title('Comparación de Clusters')
else:
    ax11.text(0.5, 0.5, 'No hay datos\nde clusters', 
             ha='center', va='center', fontsize=12)
    ax11.set_title('Comparación de Clusters')

ax12 = fig3.add_subplot(gs[1, :])
if 'Cluster' in df_analysis.columns and 'Region' in df_analysis.columns:
    try:
        contingency = pd.crosstab(df_analysis['Region'], df_analysis['Cluster'], normalize='index')
        
        sns.heatmap(contingency, annot=True, fmt='.2f', cmap='YlOrRd', 
                    linewidths=.5, cbar_kws={'label': 'Proporción'}, ax=ax12)
        
        ax12.set_xlabel('Cluster')
        ax12.set_ylabel('Región')
        ax12.set_title('Distribución de Regiones en Clusters (Proporciones por Fila)')
        
        ax12.set_xticklabels(ax12.get_xticklabels(), rotation=0)
        ax12.set_yticklabels(ax12.get_yticklabels(), rotation=0)
    except:
        ax12.text(0.5, 0.5, 'Error al crear\nmatriz de contingencia', 
                 ha='center', va='center', fontsize=12)
        ax12.set_title('Distribución Región-Cluster')
else:
    ax12.text(0.5, 0.5, 'Variables necesarias\nno disponibles', 
             ha='center', va='center', fontsize=12)
    ax12.set_title('Distribución Región-Cluster')

plt.tight_layout()
plt.savefig('resultados_clustering.png', dpi=300, bbox_inches='tight')
print(f"{Fore.GREEN} Gráfico 3 guardado: resultados_clustering.png")
plt.show()

print_section("3.2 Análisis de Perfiles de Clusters")

if clusters_df is not None and not clusters_df.empty:
    print(f"\n{Fore.CYAN}Perfiles de Clusters (Resumen Tabular):")
    
    cluster_table = []
    for _, row in clusters_df.iterrows():
        cluster_table.append([
            f"Cluster {int(row['Cluster'])}",
            f"{int(row['Empresas']):,}",
            f"{row['% Total']:.1f}%",
            row['Región Predominante'][:20],
            f"{row['% Región']:.1f}%",
            row['Tipo Predominante'][:20],
            f"{row['% Tipo']:.1f}%",
            f"{row['Año Promedio']:.0f}",
            row['Estado Predominante'][:15]
        ])
    
    headers = ['Cluster', 'Empresas', '% Total', 'Región Pred', '% Reg', 
               'Tipo Pred', '% Tipo', 'Año Prom', 'Estado Pred']
    
    print(f"\n{tabulate(cluster_table, headers=headers, tablefmt='grid')}")
    
    fig_table, ax_table = plt.subplots(figsize=(14, len(cluster_table) * 0.5 + 2))
    ax_table.axis('tight')
    ax_table.axis('off')
    
    table = ax_table.table(cellText=cluster_table, colLabels=headers, 
                          cellLoc='center', loc='center',
                          colColours=['#4C72B0'] * len(headers))
    
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    plt.title('Perfiles de Clusters Identificados', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('tabla_perfiles_clusters.png', dpi=300, bbox_inches='tight')
    print(f"{Fore.GREEN} Tabla de perfiles guardada: tabla_perfiles_clusters.png")
    plt.show()
else:
    print(f"{Fore.YELLOW} No hay datos de clusters para mostrar")


print_header("VISUALIZACIÓN DE RESULTADOS DE CLASIFICACIÓN", 80)

print_section("4.1 Importancia de Características")

if importance_df is not None and not importance_df.empty:
    fig4, axes4 = plt.subplots(1, 2, figsize=(16, 8))
    fig4.suptitle('Análisis del Modelo de Clasificación - Random Forest', fontsize=16, fontweight='bold')
    
    ax13 = axes4[0]
    
    importance_sorted = importance_df.sort_values('Importancia', ascending=True)
    
    colors_imp = cm.Blues(np.linspace(0.3, 0.9, len(importance_sorted)))
    bars = ax13.barh(range(len(importance_sorted)), importance_sorted['Importancia'], 
                    color=colors_imp, edgecolor='black')
    
    ax13.set_yticks(range(len(importance_sorted)))
    ax13.set_yticklabels(importance_sorted['Variable'])
    ax13.set_xlabel('Importancia Relativa')
    ax13.set_title('Importancia de Características para Predecir Tipo de Sociedad')
    
    for i, (bar, importance) in enumerate(zip(bars, importance_sorted['Importancia'])):
        ax13.text(importance + 0.01, bar.get_y() + bar.get_height()/2, 
                 f'{importance*100:.1f}%', va='center', fontsize=10)
    
    ax14 = axes4[1]
    
    importance_sorted = importance_df.sort_values('Importancia', ascending=False)
    importance_sorted['Importancia_Pct'] = importance_sorted['Importancia'] * 100
    
    colors_bar = cm.Paired(np.linspace(0, 1, len(importance_sorted)))
    
    bars2 = ax14.barh(range(len(importance_sorted)), importance_sorted['Importancia_Pct'], 
                     color=colors_bar, edgecolor='black')
    
    ax14.set_yticks(range(len(importance_sorted)))
    ax14.set_yticklabels(importance_sorted['Variable'])
    ax14.set_xlabel('Importancia (%)')
    ax14.set_title('Distribución de Importancia de Características')
    ax14.invert_yaxis()  
    
    for bar, pct in zip(bars2, importance_sorted['Importancia_Pct']):
        width = bar.get_width()
        ax14.text(width + 1, bar.get_y() + bar.get_height()/2, 
                 f'{pct:.1f}%', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('importancia_caracteristicas.png', dpi=300, bbox_inches='tight')
    print(f"{Fore.GREEN} Gráfico 4 guardado: importancia_caracteristicas.png")
    plt.show()
else:
    print(f"{Fore.YELLOW} No hay datos de importancia de características")

print_section("4.2 Predicciones por Región")

try:
    pred_df = pd.read_csv('predicciones_por_region.csv')
    
    if not pred_df.empty:
        fig5, axes5 = plt.subplots(1, 2, figsize=(16, 8))
        fig5.suptitle('Predicciones de Tipo de Sociedad por Región', fontsize=16, fontweight='bold')
        
        ax15 = axes5[0]
        
        heatmap_data = pred_df.pivot_table(index='Region', 
                                          columns='Tipo Más Probable', 
                                          values='Probabilidad', 
                                          aggfunc='first')
        
        sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn',
                    linewidths=.5, cbar_kws={'label': 'Probabilidad (%)'}, ax=ax15)
        
        ax15.set_xlabel('Tipo de Sociedad Predicho')
        ax15.set_ylabel('Región')
        ax15.set_title('Probabilidad del Tipo Más Probable por Región')
        ax15.tick_params(axis='x', rotation=45)
        ax15.tick_params(axis='y', rotation=0)
        
        ax16 = axes5[1]
        
        top_pred = pred_df.nlargest(10, 'Probabilidad')
        
        x = np.arange(len(top_pred))
        width = 0.35
        
        bars1 = ax16.bar(x - width/2, top_pred['Probabilidad'], width,
                        label='Tipo Más Probable', color='green', alpha=0.7)
        bars2 = ax16.bar(x + width/2, top_pred['Probabilidad 2do'], width,
                        label='Segundo Más Probable', color='orange', alpha=0.7)
        
        ax16.set_xlabel('Región')
        ax16.set_ylabel('Probabilidad (%)')
        ax16.set_title('Top 10 Regiones con Mayor Probabilidad de Predicción')
        ax16.set_xticks(x)
        ax16.set_xticklabels(top_pred['Region'], rotation=45, ha='right')
        ax16.legend()
        ax16.grid(True, alpha=0.3, axis='y')
        
        for i, row in top_pred.iterrows():
            idx = list(top_pred.index).index(i)
            ax16.text(idx - width/2, row['Probabilidad'] + 1, 
                     row['Tipo Más Probable'][:15], ha='center', va='bottom', 
                     fontsize=8, rotation=90)
            ax16.text(idx + width/2, row['Probabilidad 2do'] + 1, 
                     row['Segundo Más Probable'][:15], ha='center', va='bottom', 
                     fontsize=8, rotation=90)
        
        plt.tight_layout()
        plt.savefig('predicciones_por_region.png', dpi=300, bbox_inches='tight')
        print(f"{Fore.GREEN} Gráfico 5 guardado: predicciones_por_region.png")
        plt.show()
        
except Exception as e:
    print(f"{Fore.YELLOW} Predicciones por región no encontradas: {str(e)[:50]}")


print_header("VISUALIZACIÓN DE CONTRASTE DE HIPÓTESIS", 80)

print_section("5.1 Resultados de Prueba Chi-Cuadrado")

try:
    chi2_df = pd.read_csv('resultados_chi_cuadrado.csv')
    
    if not chi2_df.empty:
        fig6, axes6 = plt.subplots(1, 2, figsize=(16, 8))
        fig6.suptitle('Resultados del Contraste de Hipótesis - Prueba Chi-Cuadrado', 
                     fontsize=16, fontweight='bold')
        
        ax17 = axes6[0]
        
        chi2_stat = chi2_df.iloc[0]['chi2_statistic']
        p_value = chi2_df.iloc[0]['p_value']
        cramers_v = chi2_df.iloc[0]['cramers_v']
        significance = chi2_df.iloc[0]['significance']
        alpha = 0.05
        
        stats_labels = ['χ² Estadístico', 'p-valor', 'V de Cramer']
        stats_values = [chi2_stat, p_value, cramers_v]
        colors_stats = ['steelblue', 'coral', 'green']
        
        bars = ax17.bar(stats_labels, stats_values, color=colors_stats, alpha=0.7, edgecolor='black')
        ax17.set_ylabel('Valor')
        ax17.set_title('Estadísticos de la Prueba Chi-Cuadrado')
        ax17.grid(True, alpha=0.3, axis='y')
        
        ax17.axhline(y=alpha, color='red', linestyle='--', linewidth=2, label=f'α = {alpha}')
        
        for bar, value in zip(bars, stats_values):
            height = bar.get_height()
            ax17.text(bar.get_x() + bar.get_width()/2., height + max(stats_values)*0.02,
                     f'{value:.4f}', ha='center', va='bottom', fontsize=10)
        
        ax17.legend()
        
        ax18 = axes6[1]
        
        decision_labels = ['p-valor', 'Nivel α']
        decision_values = [p_value, alpha]
        
        colors_decision = ['red' if p_value < alpha else 'green', 'gray']
        
        bars_decision = ax18.bar(decision_labels, decision_values, 
                                color=colors_decision, alpha=0.7, edgecolor='black')
        
        ax18.set_ylabel('Valor')
        ax18.set_title('Decisión Estadística: p-valor vs Nivel de Significancia')
        ax18.grid(True, alpha=0.3, axis='y')
        
        ax18.axhline(y=alpha, color='red', linestyle='--', linewidth=2, alpha=0.5)
        
        for bar, value in zip(bars_decision, decision_values):
            height = bar.get_height()
            ax18.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                     f'{value:.4f}', ha='center', va='bottom', fontsize=10)
        
        conclusion_text = f"Decisión: {'RECHAZAR H₀' if significance else 'NO RECHAZAR H₀'}\n"
        conclusion_text += f"Existen {'diferencias' if significance else 'no hay diferencias'} significativas\n"
        conclusion_text += f"Fuerza asociación: {chi2_df.iloc[0]['strength_of_association']}"
        
        ax18.text(0.5, -0.3, conclusion_text, ha='center', va='top', 
                 transform=ax18.transAxes, fontsize=11, fontweight='bold',
                 bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8))
        
        plt.tight_layout()
        plt.savefig('resultados_chi_cuadrado.png', dpi=300, bbox_inches='tight')
        print(f"{Fore.GREEN} Gráfico 6 guardado: resultados_chi_cuadrado.png")
        plt.show()
        
except Exception as e:
    print(f"{Fore.YELLOW} Resultados de chi-cuadrado no encontrados: {str(e)[:50]}")


print_header("CREACIÓN DE REPORTE VISUAL", 80)

print_section("6.1 Generación de Reporte HTML")

report_html = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Reporte de Análisis RUES - Visualización de Resultados</title>
    <style>
        body { 
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
            margin: 0; 
            padding: 20px; 
            background-color: #f5f5f5; 
            color: #333; 
        }
        .container { 
            max-width: 1200px; 
            margin: 0 auto; 
            background-color: white; 
            padding: 30px; 
            border-radius: 10px; 
            box-shadow: 0 0 20px rgba(0,0,0,0.1); 
        }
        .header { 
            text-align: center; 
            padding: 20px; 
            background: linear-gradient(135deg, #2c3e50, #4a6491); 
            color: white; 
            border-radius: 10px; 
            margin-bottom: 30px; 
        }
        .header h1 { 
            margin: 0; 
            font-size: 2.5em; 
        }
        .header p { 
            margin: 10px 0 0; 
            font-size: 1.2em; 
            opacity: 0.9; 
        }
        .section { 
            margin: 40px 0; 
            padding: 25px; 
            background-color: #f8f9fa; 
            border-radius: 8px; 
            border-left: 5px solid #3498db; 
        }
        .section h2 { 
            color: #2c3e50; 
            margin-top: 0; 
            border-bottom: 2px solid #3498db; 
            padding-bottom: 10px; 
        }
        .image-container { 
            text-align: center; 
            margin: 20px 0; 
            padding: 15px; 
            background-color: white; 
            border-radius: 8px; 
            box-shadow: 0 2px 10px rgba(0,0,0,0.05); 
        }
        .image-container img { 
            max-width: 100%; 
            height: auto; 
            border: 1px solid #ddd; 
            border-radius: 5px; 
        }
        .image-caption { 
            margin-top: 10px; 
            font-style: italic; 
            color: #666; 
        }
        .stats-grid { 
            display: grid; 
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); 
            gap: 20px; 
            margin: 20px 0; 
        }
        .stat-card { 
            background: white; 
            padding: 20px; 
            border-radius: 8px; 
            text-align: center; 
            box-shadow: 0 2px 5px rgba(0,0,0,0.1); 
            transition: transform 0.3s; 
        }
        .stat-card:hover { 
            transform: translateY(-5px); 
            box-shadow: 0 5px 15px rgba(0,0,0,0.1); 
        }
        .stat-value { 
            font-size: 2.5em; 
            font-weight: bold; 
            color: #2c3e50; 
            margin: 10px 0; 
        }
        .stat-label { 
            font-size: 1em; 
            color: #7f8c8d; 
        }
        .conclusion-box { 
            background-color: #e8f6f3; 
            padding: 25px; 
            border-radius: 8px; 
            margin: 20px 0; 
            border-left: 5px solid #1abc9c; 
        }
        .conclusion-box h3 { 
            color: #16a085; 
            margin-top: 0; 
        }
        .file-list { 
            list-style-type: none; 
            padding: 0; 
        }
        .file-list li { 
            padding: 10px; 
            margin: 5px 0; 
            background-color: white; 
            border-radius: 5px; 
            border-left: 4px solid #3498db; 
        }
        .footer { 
            text-align: center; 
            margin-top: 40px; 
            padding: 20px; 
            color: #7f8c8d; 
            border-top: 1px solid #eee; 
        }
        @media (max-width: 768px) {
            .container { 
                padding: 15px; 
            }
            .header h1 { 
                font-size: 2em; 
            }
            .stats-grid { 
                grid-template-columns: 1fr; 
            }
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1> Reporte de Análisis RUES</h1>
            <p>Visualización Completa de Resultados</p>
            <p>Fecha de generación: """ + pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S') + """</p>
        </div>
        
        <div class="section">
            <h2> Resumen Ejecutivo</h2>
            <div class="stats-grid">
                <div class="stat-card">
                    <div class="stat-value">""" + f"{len(df_analysis):,}" + """</div>
                    <div class="stat-label">Empresas Analizadas</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">""" + f"{df_analysis['Region'].nunique() if 'Region' in df_analysis.columns else 'N/A'}" + """</div>
                    <div class="stat-label">Regiones Identificadas</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">""" + f"{df_analysis['Tipo de Sociedad'].nunique() if 'Tipo de Sociedad' in df_analysis.columns else 'N/A'}" + """</div>
                    <div class="stat-label">Tipos de Sociedad</div>
                </div>
"""

if clusters_df is not None and not clusters_df.empty:
    num_clusters = len(clusters_df)
    report_html += f"""
                <div class="stat-card">
                    <div class="stat-value">{num_clusters}</div>
                    <div class="stat-label">Clusters Identificados</div>
                </div>
    """

report_html += """
            </div>
        </div>
        
        <div class="section">
            <h2> Distribución General del Dataset</h2>
            <div class="image-container">
                <img src="distribucion_variables_principales.png" alt="Distribución de Variables Principales">
                <div class="image-caption">Figura 1: Distribución de variables principales del dataset RUES</div>
            </div>
            
            <div class="image-container">
                <img src="analisis_geografico_detallado.png" alt="Análisis Geográfico Detallado">
                <div class="image-caption">Figura 2: Análisis geográfico detallado por región</div>
            </div>
        </div>
"""

if clusters_df is not None and not clusters_df.empty:
    report_html += """
        <div class="section">
            <h2> Segmentación por Clustering</h2>
            <div class="image-container">
                <img src="resultados_clustering.png" alt="Resultados de Clustering">
                <div class="image-caption">Figura 3: Resultados del análisis de clustering</div>
            </div>
            
            <div class="conclusion-box">
                <h3>Perfiles de Clusters Identificados</h3>
                <p>Se identificaron """ + str(len(clusters_df)) + """ segmentos naturales de empresas con características distintivas:</p>
                <ul>
    """
    
    for _, row in clusters_df.head(3).iterrows():  
        report_html += f"""
                    <li><strong>Cluster {int(row['Cluster'])}</strong>: {int(row['Empresas']):,} empresas ({row['% Total']:.1f}%) - 
                    Predomina en {row['Región Predominante']} con {row['Tipo Predominante']}</li>
        """
    
    report_html += """
                </ul>
            </div>
        </div>
    """

if importance_df is not None and not importance_df.empty:
    report_html += """
        <div class="section">
            <h2> Modelo Predictivo - Random Forest</h2>
            <div class="image-container">
                <img src="importancia_caracteristicas.png" alt="Importancia de Características">
                <div class="image-caption">Figura 4: Importancia de características para predecir tipo de sociedad</div>
            </div>
            
            <div class="conclusion-box">
                <h3>Características Más Predictivas</h3>
                <p>Las variables más importantes para predecir el tipo de sociedad son:</p>
                <ol>
    """
    
    for _, row in importance_df.head(3).iterrows():  
        report_html += f"""
                    <li><strong>{row['Variable']}</strong>: {row['Importancia']*100:.1f}% de importancia</li>
        """
    
    report_html += """
                </ol>
            </div>
        </div>
    """

try:
    chi2_df = pd.read_csv('resultados_chi_cuadrado.csv')
    if not chi2_df.empty:
        significance = "SÍ" if chi2_df.iloc[0]['significance'] else "NO"
        strength = chi2_df.iloc[0]['strength_of_association']
        
        report_html += f"""
        <div class="section">
            <h2> Contraste de Hipótesis Estadístico</h2>
            <div class="image-container">
                <img src="resultados_chi_cuadrado.png" alt="Resultados Chi-Cuadrado">
                <div class="image-caption">Figura 5: Resultados del contraste de hipótesis</div>
            </div>
            
            <div class="conclusion-box">
                <h3>Conclusión Estadística</h3>
                <p><strong>¿Existen diferencias significativas en la distribución de tipos de sociedad entre regiones?</strong></p>
                <p style="font-size: 1.2em; font-weight: bold; color: {'#e74c3c' if chi2_df.iloc[0]['significance'] else '#27ae60'}">
                    {significance} - {chi2_df.iloc[0]['significance'] and 'Se rechaza H₀' or 'No se rechaza H₀'}
                </p>
                <p><strong>Fuerza de la asociación:</strong> {strength}</p>
                <p><strong>p-valor:</strong> {chi2_df.iloc[0]['p_value']:.6f}</p>
            </div>
        </div>
        """
except:
    pass

report_html += """
        <div class="section">
            <h2> Recomendaciones Estratégicas</h2>
            <div class="conclusion-box">
                <h3>Para la Cámara de Comercio</h3>
                <ul>
                    <li><strong>Políticas regionales diferenciadas:</strong> Diseñar estrategias específicas para cada perfil de cluster identificado</li>
                    <li><strong>Automatización de procesos:</strong> Implementar el modelo predictivo para clasificación automática de nuevas empresas</li>
                    <li><strong>Optimización de recursos:</strong> Priorizar regiones con alta concentración de tipos específicos de sociedad</li>
                    <li><strong>Monitoreo continuo:</strong> Establecer un sistema de seguimiento para detectar cambios en los patrones identificados</li>
                    <li><strong>Investigación adicional:</strong> Profundizar en las regiones con diferencias significativas para entender los factores subyacentes</li>
                </ul>
            </div>
        </div>
        
        <div class="section">
            <h2> Archivos Generados</h2>
            <p>Los siguientes archivos han sido generados durante el análisis:</p>
            <ul class="file-list">
                <li> <strong>distribucion_variables_principales.png</strong> - Distribución general del dataset</li>
                <li> <strong>analisis_geografico_detallado.png</strong> - Análisis geográfico por región</li>
                <li> <strong>resultados_clustering.png</strong> - Resultados de segmentación por clustering</li>
                <li> <strong>importancia_caracteristicas.png</strong> - Importancia de características del modelo predictivo</li>
                <li> <strong>resultados_chi_cuadrado.png</strong> - Resultados del contraste de hipótesis</li>
                <li> <strong>tabla_perfiles_clusters.png</strong> - Tabla de perfiles de clusters</li>
                <li> <strong>rues_analisis_completo.csv</strong> - Dataset completo con análisis</li>
                <li> <strong>resultados_clustering.csv</strong> - Resultados detallados de clustering</li>
                <li> <strong>importancia_caracteristicas.csv</strong> - Importancia de características</li>
            </ul>
        </div>
        
        <div class="footer">
            <p>Reporte generado automáticamente por el sistema de análisis RUES</p>
            <p>© 2024 - Análisis de Datos Empresariales | Todos los derechos reservados</p>
        </div>
    </div>
</body>
</html>
"""

with open('reporte_visual_rues.html', 'w', encoding='utf-8') as f:
    f.write(report_html)

print(f"{Fore.GREEN} Reporte HTML guardado: reporte_visual_rues.html")


print_header("FASE 4 COMPLETADA - VISUALIZACIONES GENERADAS", 80)

print(f"\n{Fore.CYAN}RESUMEN DE ARCHIVOS GENERADOS:")

print(f"\n{Fore.MAGENTA}1. GRÁFICOS ESTÁTICOS (PNG):")
print(f"{Fore.WHITE}• distribucion_variables_principales.png - Distribución general (6 subgráficos)")
print(f"{Fore.WHITE}• analisis_geografico_detallado.png - Análisis por región (2 subgráficos)")
print(f"{Fore.WHITE}• resultados_clustering.png - Segmentación por clusters (4 subgráficos)")
print(f"{Fore.WHITE}• importancia_caracteristicas.png - Importancia de predictores (2 subgráficos)")
print(f"{Fore.WHITE}• tabla_perfiles_clusters.png - Tabla de perfiles")
print(f"{Fore.WHITE}• predicciones_por_region.png - Predicciones por región (si disponible)")
print(f"{Fore.WHITE}• resultados_chi_cuadrado.png - Resultados contraste hipótesis (2 subgráficos)")

print(f"\n{Fore.MAGENTA}2. REPORTES Y DOCUMENTACIÓN:")
print(f"{Fore.WHITE}• reporte_visual_rues.html - Reporte HTML ejecutivo con visualizaciones")

print(f"\n{Fore.MAGENTA}3. DATOS ANALIZADOS:")
print(f"{Fore.WHITE}• rues_analisis_completo.csv - Dataset con análisis completo")

print(f"\n{Fore.GREEN} Todas las visualizaciones han sido generadas exitosamente")
print(f"{Fore.YELLOW} Puedes abrir 'reporte_visual_rues.html' en tu navegador para ver el reporte completo")
print(f"{Fore.YELLOW} Los gráficos están guardados como imágenes PNG de alta calidad (300 DPI)")


print(f"\n{Fore.LIGHTBLACK_EX}{'='*80}")
print(f"{Fore.GREEN}{'REPRESENTACIÓN VISUAL COMPLETADA'.center(80)}")
print(f"{Fore.LIGHTBLACK_EX}{'='*80}")
